In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import os

os.makedirs("outputs/figures", exist_ok=True)

In [ ]:
paths = []
with open('paths.txt', 'r') as file:
    for line in file.readlines():
        paths.append(line.strip())

In [ ]:
fig, ax = plt.subplots()
for i in range(len(paths)):
    path = paths[i].split("/")[1:]
    save_path = f"outputs/{path[1]}/{path[3]}"

    df = pd.read_csv(f'{save_path}/data.csv')

    if df["tasks_complete"].iloc[-1] < 72:
        continue

    ax.plot(df["normalized_datetime"], df["tasks_complete"], label=str(df.num_cores.iloc[0]))

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Jobs Complete")
    ax.set_title(f"All Runs Together")
plt.savefig("outputs/figures/all_runs_combined.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
for num_cores in [4, 8, 16, 32, 64]:
    fig, ax = plt.subplots()
    for i in range(len(paths)):
        path = paths[i].split("/")[1:]
        save_path = f"outputs/{path[1]}/{path[3]}"

        df = pd.read_csv(f'{save_path}/data.csv')

        if df["tasks_complete"].iloc[-1] < 72:
            continue
        if df["num_cores"].iloc[0] != num_cores:
            continue

        ax.plot(df["normalized_datetime"], df["tasks_complete"], label=str(df.num_cores.iloc[0]))

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Jobs Complete")
        ax.set_title(f"{num_cores} Core Runs")
    plt.savefig(f"outputs/figures/{num_cores}_core_runs.pdf", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig, ax = plt.subplots()
for i in range(len(paths)):
    path = paths[i].split("/")[1:]
    save_path = f"outputs/{path[1]}/{path[3]}"

    df = pd.read_csv(f'{save_path}/data.csv')

    if df["tasks_complete"].iloc[-1] < 72:
        continue

    num_cores = df.num_cores.iloc[0]
    cores_map = {
            4 : "red",
            8 : "blue",
            16: "green",
            32: "gray",
            64: "orange",
        }
    ax.plot(df["normalized_datetime"], df["tasks_complete"], label=str(num_cores), color=cores_map[num_cores])

# Create one legend entry for each color
legend_handles = [
    Line2D([0], [0], color=color, lw=2, label=f"{cores} cores")
    for cores, color in cores_map.items()
]

ax.legend(handles=legend_handles, title="CPU Cores")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Jobs Complete")
ax.set_title(f"All Runs (sorted colors)")
plt.savefig(f"outputs/figures/all_runs_color_sorted.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots()
for num_cores in [4, 8, 16, 32, 64]:
    dfs = []
    for path in paths:
        path_parts = path.split("/")[1:]
        save_path = f"outputs/{path_parts[1]}/{path_parts[3]}"

        df = pd.read_csv(f'{save_path}/data.csv')
        if df["num_cores"].iloc[0] != num_cores:
            continue
        if df["tasks_complete"].iloc[-1] < 72:
            continue

        df = df[df["job_id"] <= 72]

        dfs.append(df)
    
    all_data = pd.concat(dfs, ignore_index=True)

    avg_trajectory = (
        all_data.groupby("tasks_complete")["normalized_datetime"]
        .agg(["mean", "std"])
        .reset_index()
    )

    # -------------------------------------------------------------
    # 3. Plot average completion line (X = Time, Y = Jobs Complete)
    # -------------------------------------------------------------
    cores_map = {
        4 : "red",
        8 : "blue",
        16: "green",
        32: "gray",
        64: "orange",
    }
    plt.plot(
        avg_trajectory["mean"],
        avg_trajectory["tasks_complete"],
        linewidth=2.5,
        label=str(num_cores),
        color=cores_map[num_cores]
    )

    # Add ±1 Standard Deviation band across runs
    # plt.fill_betweenx(
    #     avg_trajectory["tasks_complete"],
    #     avg_trajectory["mean"] - avg_trajectory["std"],
    #     avg_trajectory["mean"] + avg_trajectory["std"],
    #     color=cores_map[num_cores],
    #     alpha=0.15,
    #     label="±1 Std Dev",
    # )

ax.set_xlabel("Time Since Start (Seconds)")
ax.set_ylabel("Number of Jobs Completed")
ax.set_title("Average Job Completion Across Core Count")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(loc="lower right", title="CPU Cores")
plt.tight_layout()
plt.savefig(f"outputs/figures/average_completion_vs_cores.pdf", dpi=300, bbox_inches='tight')
plt.show()